## Predicting ticket priority
We are building a model that does this automatically:

- Customer writes: "Our entire engineering team has been 
locked out of the platform since this morning. We have 
a product launch tomorrow and cannot access any of our 
data. This is costing us thousands of dollars per hour."

    - Model reads it → outputs: CRITICAL

- Customer writes: "hey just wondering if you guys might 
be adding dark mode sometime? would be kinda nice lol"

    - Model reads it → outputs: LOW

- Unlike ticket type where the words directly signal the category ("refund" → Refund Request), priority is about tone, urgency, and business impact — not the topic.
The same technical issue can be:

    - Critical: "entire company blocked, launch tomorrow"
    - Low: "minor glitch, not affecting work"

- BERT needs to read emotional intensity and business impact language, not just topic keywords.

- There fore we will use Urgency score:
    - words like "immediately","entire team", "urgent" are directly indicating high priority. COunting them gibes the model an explicit unrgency signal along side Bert's semantic uunderstanding

In [32]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [33]:
df = pd.read_csv("../Dataset/cleaned_tickets.csv")
df.head()

,Product Purchased,Ticket Type,Ticket Subject,Ticket Status,Ticket Priority,Ticket Channel,combine_text,ticket_type_encoded,ticket_priority_encoded
0,Massage Gun Pro,Technical Issue,Something went wrong after the latest update,Resolved,Medium,Social Media,something went wrong after the latest update i...,4,3
1,LaptopX 15,Product Inquiry,I have a few questions about a product,In Progress,Medium,Social Media,i have a few questions about a product before ...,2,3
2,SoundWave Earbuds,Technical Issue,Something went wrong after the latest update,Open,High,Social Media,something went wrong after the latest update i...,4,1
3,KitchenBlast Blender,Product Inquiry,Product question from a potential customer,Resolved,Medium,Chat,product question from a potential customer i'm...,2,3
4,Massage Gun Pro,Refund Request,Help with processing a refund,Resolved,Low,Chat,help with processing a refund i'll keep this b...,3,2


In [34]:
X = df['combine_text']
y = df['ticket_priority_encoded']

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [78]:
## loading encoder pkl file to encode the training and testing data
encoder = joblib.load("../models/sentence_encoder.pkl")

In [ ]:
print("Encoding the training data...")
X_train_embeddings = encoder.encode(
    X_train.tolist(),
    show_progress_bar = True,
    batch_size = 32
)

print("encoding test data..")
X_test_embeddings  = encoder.encode(
    X_test.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("train embeddings shape: ", X_train_embeddings.shape)
print("test_embeddings shape ",X_test_embeddings.shape)

Encoding the training data...


Batches: 100%|██████████| 113/113 [00:35<00:00,  3.16it/s]


encoding test data


Batches: 100%|██████████| 29/29 [00:08<00:00,  3.32it/s]

train embeddings shape:  (3600, 384)
test_embeddings shape  (900, 384)


### Testing first model: logistic regression

In [80]:
model1 = LogisticRegression(max_iter=2000)
model1.fit(X_train_embeddings,y_train)
print("Model 1 trained")

Model 1 trained


In [81]:
le_priority = joblib.load("../models/le_priority.pkl")
print(le_priority)

LabelEncoder()


In [82]:
y_pred = model1.predict(X_test_embeddings)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\n")
print(classification_report(y_test, y_pred, target_names=le_priority.classes_))  
## target_names gives back the actual name to the encodes priorities

Accuracy: 0.42777777777777776


              precision    recall  f1-score   support

    Critical       0.81      0.35      0.48       110
        High       0.25      0.09      0.13       193
         Low       0.42      0.25      0.31       256
      Medium       0.42      0.78      0.55       341

    accuracy                           0.43       900
   macro avg       0.48      0.37      0.37       900
weighted avg       0.43      0.43      0.38       900



### testing second model: Ranodm Forest

In [83]:
model2 = RandomForestClassifier()
model2.fit(X_train_embeddings,y_train)
print("Model 2 trained")

Model 2 trained


In [84]:
y_pred2 = model2.predict(X_test_embeddings)

print("Accuracy:", accuracy_score(y_test, y_pred2))
print("\n")
print(classification_report(y_test, y_pred2, target_names=le_priority.classes_))

Accuracy: 0.3988888888888889


              precision    recall  f1-score   support

    Critical       0.83      0.26      0.40       110
        High       0.22      0.09      0.12       193
         Low       0.37      0.28      0.32       256
      Medium       0.41      0.71      0.52       341

    accuracy                           0.40       900
   macro avg       0.46      0.33      0.34       900
weighted avg       0.41      0.40      0.36       900



### testing third model: Support vector machine (SVM) classifier

In [87]:
from sklearn.svm import SVC

model3 = SVC(kernel='rbf', C=1.0)
model3.fit(X_train_embeddings, y_train)

y_pred3 = model3.predict(X_test_embeddings)
print("SVM Accuracy:", accuracy_score(y_test, y_pred3))
print(classification_report(y_test, y_pred3, 
      target_names=le_priority.classes_))

SVM Accuracy: 0.44333333333333336
              precision    recall  f1-score   support

    Critical       0.95      0.35      0.51       110
        High       0.25      0.05      0.09       193
         Low       0.66      0.17      0.27       256
      Medium       0.41      0.90      0.56       341

    accuracy                           0.44       900
   macro avg       0.57      0.37      0.36       900
weighted avg       0.51      0.44      0.37       900



### testing another approach: combine features from description and use them to train model

In [85]:
# Feature 1: Description length (longer = more detailed = needs higher priority)
df['desc_length'] = df['combine_text'].str.len()

# Feature 2: Exclamation marks (emotional intensity)
df['exclamation_count'] = df['combine_text'].str.count('!')

# Feature 3: Question marks (uncertainty/confusion)
df['question_count'] = df['combine_text'].str.count(r'\?')

# Feature 5: Ticket type encoded
# done before

In [86]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
extra_features_scaled = scaler.fit_transform(
    df[['desc_length', 'exclamation_count', 
        'question_count',
        'ticket_type_encoded']]
)

X_train_final = np.hstack([
    X_train_embeddings,
    extra_features_scaled[:len(X_train)]
])

X_test_final = np.hstack([
    X_test_embeddings,
    extra_features_scaled[len(X_train):]
])

In [88]:
model4 = LogisticRegression(max_iter=2000)
model4.fit(X_train_final, y_train)

y_pred4 = model4.predict(X_test_final)
print("Accuracy:", accuracy_score(y_test, y_pred4))
print(classification_report(y_test, y_pred4, 
      target_names=le_priority.classes_))

Accuracy: 0.42444444444444446
              precision    recall  f1-score   support

    Critical       0.81      0.35      0.49       110
        High       0.25      0.09      0.14       193
         Low       0.42      0.27      0.33       256
      Medium       0.42      0.75      0.54       341

    accuracy                           0.42       900
   macro avg       0.47      0.37      0.37       900
weighted avg       0.43      0.42      0.39       900



## Conclusion

After testing Logistic Regression, Random Forest, SVM, and feature engineering approaches, all models converged around 41-44% accuracy. Text alone is insufficient for reliable priority prediction due to tone-mismatched tickets.

Decision: Rule-based priority system can be tested, the one giving the best accuracy and classification report will be implemented

In [75]:
def assign_priority(ticket_type, text, channel):
    text = text.lower()
    
    # Critical keywords — check first, highest priority
    critical_keywords = ['team', 'deadline', 'tomorrow', 
                         'urgent', 'immediately', 'everyone',
                         'production', 'down', 'emergency']
    
    if any(word in text for word in critical_keywords):
        return 'Critical'
    
    # Channel bump
    if channel == 'email':
        return 'Critical'      
    
    # Ticket type defaults
    type_priority = {
        'Technical Issue': 'High',    
        'Refund Request': 'Medium',     
        'Billing Inquiry': 'Medium',    
        'Account Access': 'Medium',     
        'Product Inquiry': 'Low'     
    }
    
    return type_priority.get(ticket_type, 'Medium')

In [76]:
# Test cases
print(assign_priority('Technical Issue', 
      'the entire team cannot access the platform', 'Email'))
print(assign_priority('Refund Request', 
      'i want my money back please', 'Chat'))
print(assign_priority('Billing Inquiry', 
      'deadline is tomorrow and payment failed', 'Phone'))

Critical
Medium
Critical


In [89]:
# Apply rule-based function to entire dataset
df['predicted_priority'] = df.apply(
    lambda row: assign_priority(
        row['Ticket Type'], 
        row['combine_text'], 
        row['Ticket Channel']
    ), axis=1
)

# Compare with actual priority
print("Rule-based Accuracy:", 
      accuracy_score(df['Ticket Priority'], df['predicted_priority']))
print(classification_report(df['Ticket Priority'], df['predicted_priority']))

Rule-based Accuracy: 0.31822222222222224
              precision    recall  f1-score   support

    Critical       0.20      0.21      0.21       602
        High       0.23      0.20      0.21      1013
         Low       0.30      0.20      0.24      1239
      Medium       0.39      0.52      0.45      1646

    accuracy                           0.32      4500
   macro avg       0.28      0.28      0.28      4500
weighted avg       0.31      0.32      0.31      4500



# Final conclusion
- we tried multiple approaches ranging from logistic regression to rule based apprach
- Logistic regression gave 42.4%
- Random forest: 39.7%
- SVM: 44.3%
- feature engineerid logistic: 42.4%
- Rule based: 31.8%

- the best out of these is SVM, we will use hyper parameter to further get a slightly better accuracy, but it will only make slight difference
- we will use confidence level in the final streamlit app
    - If confidence is high → show the prediction boldly
    - If confidence is low → show "Uncertain — manual review recommended"

## SVM hyper parameter tuning

In [90]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto', 0.001, 0.01]
}

grid_search_svm = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train_embeddings, y_train)

print("Best params:", grid_search_svm.best_params_)
print("Best CV accuracy:", grid_search_svm.best_score_)

best_svm = grid_search_svm.best_estimator_
y_pred_svm = best_svm.predict(X_test_embeddings)
print("Test accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm,
      target_names=le_priority.classes_))

Fitting 5 folds for each of 32 candidates, totalling 160 fits


d:\customer_support_ticket_classifier\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Best params: {'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}
Best CV accuracy: 0.4294444444444444
Test accuracy: 0.45111111111111113
              precision    recall  f1-score   support

    Critical       0.97      0.28      0.44       110
        High       0.00      0.00      0.00       193
         Low       0.88      0.14      0.24       256
      Medium       0.41      0.99      0.58       341

    accuracy                           0.45       900
   macro avg       0.56      0.35      0.32       900
weighted avg       0.52      0.45      0.34       900



In [92]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

base_svm = SVC(C=100, gamma='auto', kernel='rbf')
final_priority_model = CalibratedClassifierCV(base_svm, ensemble=False)
final_priority_model.fit(X_train_embeddings, y_train)

y_pred_final = final_priority_model.predict(X_test_embeddings)
print("Final model accuracy:", accuracy_score(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final, 
      target_names=le_priority.classes_,zero_division=0))

Final model accuracy: 0.44555555555555554
              precision    recall  f1-score   support

    Critical       0.76      0.32      0.45       110
        High       0.00      0.00      0.00       193
         Low       0.82      0.11      0.19       256
      Medium       0.41      0.99      0.58       341

    accuracy                           0.45       900
   macro avg       0.50      0.35      0.31       900
weighted avg       0.48      0.45      0.33       900



- C — Regularization strength:
  - Controls how much the model penalizes misclassifications during training.
  - Low C (0.1) → smoother decision boundary, more misclassifications allowed, less overfitting
  - High C (100) → tries to classify everything correctly, tighter boundary, risk of overfitting
  - We test a wide range because we don't know where the sweet spot is for this data.

- kernel — Decision boundary shape:
  - rbf (Radial Basis Function) — draws curved boundaries, good for complex patterns
  - linear — draws straight boundaries, faster, sometimes better on high-dimensional data like BERT embeddings
  - We test both because 384 dimensions is genuinely high-dimensional — linear sometimes wins here.

- gamma — Influence radius of each training point:
  - Only applies to rbf kernel.
  - scale — automatically calculated as 1/(n_features × variance)
  - auto — calculated as 1/n_features
  - 0.001, 0.01 — manual values, smaller gamma = smoother boundary
  - Low gamma means each point influences a wider area — better generalization. High gamma means each point only influences nearby points — can overfit.

- probability=True
  - Needed so you can call predict_proba() later in Streamlit for confidence scores.